# Qwen Image Edit Web Server (ComfyUI) — Paperspace

ComfyUI APIをバックエンドとして、Web サーバーを起動し、cloudflared で外部公開します。

**必要環境:**
 - comfyUI起動可能なDocker
 - free-A4000以上

## 1. torchのインストール

In [ ]:
!pip install -U huggingface_hub requests urllib3 charset-normalizer

import os
import sys
import subprocess
import platform

def get_system_cuda_version():
    """nvidia-smiを使用してシステムのCUDAバージョンを取得する"""
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits"]).decode()
        # CUDAバージョンを直接取得する（もしくはnvccから取得）
        cuda_out = subprocess.check_output(["nvcc", "--version"]).decode()
        for line in cuda_out.split('\n'):
            if "release" in line:
                return line.split("release ")[1].split(",")[0].replace(".", "")
    except:
        # 万が一取得失敗した場合はデフォルトを指定（例: 130）
        return "130"
    return "130"

def run_command(command):
    print(f"Executing: {command}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + command.split())

# 1. CUDAバージョンの取得
cuda_tag = f"cu{get_system_cuda_version()}"
print(f"🚀 Detected System CUDA: {cuda_tag}")

# 2. PyTorchのインストール (CUDAバージョンに合わせる)
print(f"📦 Installing PyTorch for {cuda_tag}...")
torch_index_url = f"https://download.pytorch.org/whl/{cuda_tag}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch", "torchvision", "torchaudio", "--index-url", torch_index_url])

print("\nInstallations completed successfully!")

## llama-cpp-pythonのインストール

In [ ]:
import sys
import platform
import requests
import torch

def get_compatible_wheel_url(repo="JamePeng/llama-cpp-python", search_limit=10):
    # 1. 現在の環境情報の取得
    py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
    
    # プラットフォーム正規化
    system = platform.system().lower()
    machine = platform.machine().lower()
    if system == "linux":
        plat = "linux_x86_64"
    elif system == "windows":
        plat = "win_amd64"
    else:
        plat = system

    # CUDAタグの生成 (13.0 -> cu130)
    cuda_tag = ""
    if torch.cuda.is_available() and torch.version.cuda:
        cuda_tag = f"cu{torch.version.cuda.replace('.', '')}"
    else:
        cuda_tag = "cpu"

    print(f"🚀 実行環境: Python={py_ver}, CUDA={cuda_tag}, Platform={plat}")
    print(f"📡 {repo} の過去 {search_limit} 件のリリースをスキャン中...")

    # 2. GitHub APIでリリースのリストを取得 (latestではなく/releases)
    try:
        api_url = f"https://api.github.com/repos/{repo}/releases"
        response = requests.get(api_url, params={"per_page": search_limit}, timeout=10)
        response.raise_for_status()
        releases = response.json()
    except Exception as e:
        print(f"❌ APIエラー: {e}")
        return None

    # 3. 各リリース内のAssetsを順にチェック
    for release in releases:
        assets = release.get("assets", [])
        tag_name = release.get("tag_name")
        
        for asset in assets:
            name = asset["name"]
            if not name.endswith(".whl"):
                continue
            
            # 条件判定: Pythonバージョン、プラットフォーム、CUDAタグがすべて含まれているか
            # JamePeng版の命名規則: llama_cpp_python-...[cuda]+[python]-[python]-[plat].whl
            if (py_ver in name) and (plat in name) and (cuda_tag in name):
                print(f"✨ マッチしたリリース: {tag_name}")
                return asset["browser_download_url"]

    return None

# --- 実行 ---
wheel_url = get_compatible_wheel_url()

if wheel_url:
    print(f"✅ 発見したURL: {wheel_url}")
    !pip install {wheel_url}
else:
    print("❌ 条件に一致するバイナリが見つかりませんでした。")
    print("手動でリポジトリを確認してください。")

## 2. パッケージインストール

In [ ]:
%cd /tmp

# ライブラリインストール
!apt -y remove python3-blinker
!pip install -U "flask>=3.0"  pillow huggingface_hub psutil accelerate googletrans huggingface_hub requests urllib3 charset-normalizer soundfile gguf --extra-index-url $torch_index_url

# cloudflared インストール（）
!wget -qN https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# ComfyUIインストール
#!git clone https://github.com/comfyanonymous/ComfyUI.git
#%cd /tmp/ComfyUI
#!git pull

!git clone https://github.com/Comfy-Org/ComfyUI.git -b v0.18.2
%cd ComfyUI

# 追加インストール
!pip install -r requirements.txt --extra-index-url $torch_index_url

# 出力先フォルダを永続ストレージ側にリンク
# !rm -r /tmp/ComfyUI/output
# !ln -s /storage/output /tmp/ComfyUI/

%cd /tmp/ComfyUI/custom_nodes/
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/id-fa/ComfyUI-llama-cpp_vlm.git

%cd /tmp

print("OK")

## 3. HuggingFace ログイン（任意）

ログインするとモデルダウンロードが高速化されます（レート制限緩和）。  
`HF_TOKEN` を登録しておくと自動認証されます。 
スキップも可。

In [ ]:
import os
from getpass import getpass

os.environ['HF_HOME'] = "/tmp/hf_hub/"

try:
    hf_token = os.environ.get("HF_TOKEN") or getpass("Enter HF token: ")
    from huggingface_hub import login
    login(token=hf_token)
    print("HuggingFace: HF_TOKEN で認証しました")
except Exception:
    print("HF_TOKEN が未登録です。手動ログインを試みます...")
    print("スキップする場合はこのセルの出力を無視してください。")
    try:
        from huggingface_hub import login
        login()
    except Exception:
        print("HuggingFace: 未ログイン（匿名ダウンロード）")

## 4. モデルのダウンロード

In [ ]:
!hf download Phr00t/Qwen-Image-Edit-Rapid-AIO --local-dir /tmp/hf_download --include "v23/Qwen-Rapid-AIO-NSFW-v23.safetensors"
%mv /tmp/hf_download/v23/Qwen-Rapid-AIO-NSFW-v23.safetensors /tmp/ComfyUI/models/diffusion_models/
!hf download Comfy-Org/Qwen-Image_ComfyUI --local-dir /tmp/hf_download --include "split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors"
%mv /tmp/hf_download/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors /tmp/ComfyUI/models/text_encoders/
!hf download Comfy-Org/Qwen-Image_ComfyUI --local-dir /tmp/hf_download --include "split_files/vae/qwen_image_vae.safetensors"
%mv /tmp/hf_download/split_files/vae/qwen_image_vae.safetensors /tmp/ComfyUI/models/vae/
!hf download mradermacher/Qwen2.5-VL-7B-Instruct-heretic-GGUF --local-dir /tmp/hf_download --include "Qwen2.5-VL-7B-Instruct-heretic.Q6_K.gguf"
%mv /tmp/hf_download/Qwen2.5-VL-7B-Instruct-heretic.Q6_K.gguf /tmp/ComfyUI/models/text_encoders/
!hf download mradermacher/Qwen2.5-VL-7B-Instruct-heretic-GGUF --local-dir ./hf_download --include "Qwen2.5-VL-7B-Instruct-heretic.mmproj-Q8_0.gguf"
%mv ./hf_download/Qwen2.5-VL-7B-Instruct-heretic.mmproj-Q8_0.gguf ./ComfyUI/models/text_encoders/

print("OK")

## 5. app_comfyui.py を配置

In [ ]:
%cd /tmp

import shutil

!git clone https://github.com/id-fa/simple-image-edit-with-qwen.git repo_tmp
!cp repo_tmp/server/app_comfyui.py .
!cp -r repo_tmp/server/lib .
!cp -r repo_tmp/server/comfyui_workflow .
    
print("準備完了\n")
!ls -la app_comfyui.py comfyui_workflow 2>/dev/null || echo "⚠ ファイルが見つかりません。"

## 6. LoRAのダウンロード（sample）

In [ ]:
%mkdir /tmp/LoRA

#!hf download sbeland/qwen2512-loras --local-dir /tmp/LoRA
!hf download Kokoboyaw/PanelPainter-Project --local-dir /tmp/hf_download --include "loras/v3/PanelPainter_v3_Qwen2511.safetensors"
%mv /tmp/hf_download/loras/v3/PanelPainter_v3_Qwen2511.safetensors /tmp/LoRA/

print("OK")

## 7. 設定

パスワードやプリセットを変更できます。

In [ ]:
# --- 設定 ---
PASSWORD = "password"       # 生成パスワード
PORT = 18080                 # サーバーポート
GALLERY = True              # ギャラリーモード有効化

# プリセット (空リストならボタン非表示)
PRESETS = [
    "高画質化::Enhance quality and fix artifacts.",
    "テキスト除去::Remove all overlaid text, subtitles, and credits.",
    "画像1に画像2の服を着せる::将 Image-1 中的角色穿上 Image-2 的服装",
]

COMFYUI_LOG = "/tmp/comfyui.log"
SERVER_LOG = "/tmp/server.log"

print("OK")

## 8. ComfyUI起動

In [ ]:
import subprocess, time, re, threading

# ログファイルに出力（パイプバッファ詰まり防止）
clog_fh = open(COMFYUI_LOG, "w")

# ComfyUI起動
# --use-flash-attention は使える環境のみ有効にする
comfyui_proc = subprocess.Popen(
    [
        "python", "-u", "/tmp/ComfyUI/main.py",
        "--listen",
        "--port", "6006",
        "--preview-method", "auto",
        "--lowvram","--dont-upcast-attention",
#       "--use-flash-attention",
    ],
    stdout=clog_fh, stderr=clog_fh, text=True
)

print("OK")

## 9. サーバー起動

In [ ]:
# コマンドライン引数を構築
cmd = [
    "python", "-u", "/tmp/app_comfyui.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--comfyui-url", "http://127.0.0.1:6006",
    "--comfyui-path", "/tmp/ComfyUI",
    "--password", PASSWORD,
    "--steps", "8",
]
if GALLERY:
    cmd.append("--gallery")
for preset in PRESETS:
    cmd.extend(["--preset", preset])

print(f"starting server: {' '.join(cmd)}")

# ログファイルに出力（パイプバッファ詰まり防止）
log_fh = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

# ログを監視してサーバー起動完了を待機
ready = False
deadline = time.time() + 600  # 10分タイムアウト
seen = 0
while time.time() < deadline:
    time.sleep(2)
    if server_proc.poll() is not None:
        print("ERROR: server process exited unexpectedly")
        with open(SERVER_LOG) as f:
            print(f.read())
        break
    with open(SERVER_LOG) as f:
        lines = f.readlines()
    # 新しい行を表示
    for line in lines[seen:]:
        print(line, end="")
    seen = len(lines)
    if any("server starting at" in l for l in lines):
        ready = True
        break

if not ready:
    raise RuntimeError("Server failed to start. Check server.log")

print(f"\n[server is running on port {PORT}")

## 10. cloudflared 公開

cloudflared で公開URLを生成します。
出力に表示される `https://******.trycloudflare.com` のURLにアクセスしてください。

In [ ]:
print(f"starting cloudflared tunnel...")

# cloudflared 起動 (stderr にURL出力)
tunnel_proc = subprocess.Popen(
    ["/tmp/cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

# stderr を読み続けるスレッド（バッファ詰まり防止）
tunnel_lines = []
def drain_tunnel_stderr():
    for line in iter(tunnel_proc.stderr.readline, ""):
        tunnel_lines.append(line)
threading.Thread(target=drain_tunnel_stderr, daemon=True).start()

# URL抽出を待機
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    time.sleep(1)
    for line in tunnel_lines:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            public_url = m.group(1)
            break
    if public_url:
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  Public URL: {public_url}")
    print(f"  Password:   {PASSWORD}")
    print(f"{'='*60}")
    print("30秒程待ってからアクセスしてください")
else:
    print("ERROR: cloudflared URL を取得できませんでした")
    print("tunnel log:", ''.join(tunnel_lines[-10:]))

print("バックグラウンドで実行中")

## 11. ユーティリティ

ログ確認・サーバー停止用。

In [ ]:
# プロセス表示
!ps auxw|grep /tmp
print("\n")

# サーバーログ確認（直近20行）
!tail -20 server.log
print("\n")

# comfyuiログ確認（直近20行）
!tail -20 comfyui.log

# サーバー + トンネル停止

```
server_proc.terminate()
comfyui_proc.terminate()
tunnel_proc.terminate()
log_fh.close()
clog_fh.close()
print("stopped")
```

In [ ]:
# コピペしてRUN
